# Mixed models with lme4

`sleepstudy` records reaction time for 18 subjects over 10 days of sleep deprivation.
Each subject has their own baseline reaction time and their own rate of decline, so the
rows are not independent — the assumption ordinary least squares needs. A mixed model
adds per-subject random intercepts and slopes and estimates the population effect on top
of them.

Every result below is checked against the published `lme4` values, so a cell that runs
without an error is a cell that got the right answer.

In [ ]:
library(lme4)
library(lmerTest)

str(sleepstudy)

## The naive fit

Pooling every subject into one regression treats 180 correlated rows as 180 independent
ones. The slope estimate happens to come out close, but the standard error is the part
that goes wrong.

In [ ]:
pooled <- lm(Reaction ~ Days, data = sleepstudy)
summary(pooled)$coefficients

## Random intercepts and slopes

`(Days | Subject)` gives each subject their own intercept *and* their own slope. The
fixed effects are the population averages: about 251 ms at day 0, increasing about
10.5 ms per day of sleep deprivation.

In [ ]:
fit <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy)
summary(fit)

In [ ]:
fe <- fixef(fit)
cat(sprintf("intercept = %.2f ms, slope = %.2f ms/day\n", fe[["(Intercept)"]], fe[["Days"]]))

# Published lme4 values for this model.
stopifnot(abs(fe[["(Intercept)"]] - 251.405) < 0.01)
stopifnot(abs(fe[["Days"]] - 10.467) < 0.01)
cat("fixed effects match the published fit\n")

## What the naive fit hid

The slope estimates agree, but the pooled standard error is a quarter smaller — it counts
repeated measurements on the same 18 people as if they were independent observations, and
so reports more confidence than the data supports.

In [ ]:
se_pooled <- summary(pooled)$coefficients["Days", "Std. Error"]
se_mixed  <- summary(fit)$coefficients["Days", "Std. Error"]
cat(sprintf("pooled SE = %.3f, mixed SE = %.3f (%.0f%% larger)\n",
            se_pooled, se_mixed, 100 * (se_mixed / se_pooled - 1)))
stopifnot(se_mixed > se_pooled)

## Per-subject estimates

`coef()` gives each subject's own intercept and slope — the population effect plus their
shrunk deviation from it. Subject 335 actually gets *faster* over the study; the pooled
model cannot express that at all.

In [ ]:
subject_coefs <- coef(fit)$Subject
head(subject_coefs[order(subject_coefs$Days), ], 3)

In [ ]:
library(ggplot2)

sleepstudy$fitted <- predict(fit)
ggplot(sleepstudy, aes(Days, Reaction)) +
  geom_point(size = 0.8) +
  geom_line(aes(y = fitted), colour = "steelblue") +
  facet_wrap(~Subject, ncol = 6) +
  labs(title = "Per-subject fits from the mixed model",
       x = "Days of sleep deprivation", y = "Reaction time (ms)") +
  theme_minimal(base_size = 9)

## Is the random slope earning its keep?

Compare against an intercept-only random effect. The likelihood ratio test is decisive,
so subjects really do differ in how fast they decline, not just in where they start.

In [ ]:
fit_int <- lmer(Reaction ~ Days + (1 | Subject), data = sleepstudy, REML = FALSE)
fit_ml  <- lmer(Reaction ~ Days + (Days | Subject), data = sleepstudy, REML = FALSE)
comparison <- anova(fit_int, fit_ml)
comparison

In [ ]:
p <- comparison[["Pr(>Chisq)"]][2]
cat(sprintf("random-slope LRT p = %.3g\n", p))
stopifnot(p < 1e-4)
cat("keep the random slope\n")